In [ ]:
#| default_exp game/globals

In [ ]:
#| export
from fastcore.utils import *
from fasthtml.common import *
from fasthtml.jupyter import *
#from fasthtml.jupyter import get_host
from fastlite import *
import fasthtml.components as fc
import pandas as pd
import httpx
import threading

In [ ]:
??show

In [ ]:
#| export
from monsterui.all import *


The `apsw.ThreadingViolationError` happens because APSW (which fastlite uses under the hood) checks that a connection is only used from the thread that created it. Since Starlette/uvicorn runs your route handlers in a **thread pool**, multiple request threads can hit `globalStore.db` concurrently.

This isn't really an async/await issue — it's a **shared connection across threads** issue. Here are two clean fixes:

**Option 1: Threading Lock (simplest)**

Serialize all DB access with a lock:

```python
import threading
db_lock = threading.Lock()

def ensure_user(session):
    with db_lock:
        uid = session.get('uid')
        row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
        # ... rest of db work
```

Wrap every function that touches the DB in `with db_lock:`. Simple, and perfectly fine for a game with moderate concurrency.

**Option 2: Thread-local connections**

Give each thread its own connection:

```python
import threading
_local = threading.local()

def get_db():
    if not hasattr(_local, 'db'):
        _local.db = database('hex.db')  # each thread gets its own connection
    return _local.db
```

Then use `get_db()` instead of `globalStore.db` in your route handlers. This allows true concurrent reads (especially with WAL mode).

**Which to pick?**

- **Lock** is easier — just wrap existing code, no structural changes. Fine for low-to-moderate traffic.
- **Thread-local** is better if you want concurrent reads, but you need to be careful about write conflicts.

For either approach, also consider enabling WAL mode on your database for better concurrent read performance:

```python
globalStore.db.execute("PRAGMA journal_mode=WAL")
```

I'd start with the threading lock since it's a one-line change per function and your current code structure stays the same.

In [ ]:
#| export
def appRoutes():
    global app, rt, hexserver
    #if 'hexserver' not in dir() or hexserver is None:
    if 'hexserver' not in globals() or hexserver is None:

        app = FastHTML(hdrs=Theme.violet.headers())
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver


In [ ]:
#| export
app, rt,  hexserver = appRoutes()

So I want app rt and hexserver only intiate once, but this will be shared among notebooks. how do I do this patter

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

The pattern you want is a **kernel-level singleton** — store it in the global namespace and check before creating. Your function is close, but has a bug: it checks `'hexGameServer'` but that's a local variable, so it never persists. Try this instead:

```python
def appRoutes():
    global app, rt, hexserver
    if 'hexserver' not in dir() or hexserver is None:
        app = FastHTML(hdrs=Theme.violet.headers())
        rt = app.route
        hexserver = JupyUvi(app)
    return app, rt, hexserver
```

The key change: use `global` so the variables live in the module namespace, and check for the same name you're assigning to.

But there's still the `FastHTML is not defined` error — did the imports cell (`_7b1b528f`) actually run successfully?

In [ ]:
#| export
def webMe(*c): return HTMX(*c, host='', app=app)

In [ ]:
@rt
def hello(name: str): return P(f"Hello {name}!")

webMe(Div(
    H3("Say Hello"),
    Form(
        Input(placeholder="Your name...", id='name'),
        Button("Send"),
        hx_get=hello, hx_target="#result"
    ),
    Div(id='result'),
))

In [ ]:
#| export
from HexMagic.game.data import ActiveGame, GameStorage, TerrainTemplate, Settlement, Piece

In [ ]:
#| export
from HexMagic.database import ZoomResult, GeoStorageDebugger,GeoStorage, SaveResult, LoadResult, ChunkCover, User

In [ ]:
#| export
import logging

logging.basicConfig(
    filename='base.text',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

logging.info("getting Started")


In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## Helpers

## ActiveGame

## Database

In [ ]:
#| export
globalStore = GameStorage("gameData/WebDebug.db")

In [ ]:
#| export
# Helper to ensure we have a user row
def ensure_user(session) -> int:
    if 'userid' not in session:
        session['userid'] = random.randint(0, 1_000_000)
    uid = session['userid']
    row = globalStore.db.execute("SELECT id FROM user WHERE id = ?", [uid]).fetchone()
    if not row:
        from datetime import datetime
        now = int(datetime.now().timestamp())
        globalStore.users.insert({
            'id': uid, 'username': f'player_{uid}', 'email': '', 'password': '',
            'created': now, 'sessionID': str(uid), 'activeWorld': 0
        })
    return uid



def new_game_page():
    templates = TerrainTemplate.maps  # {'bayArea': 'bayArea_map', ...}
    form = Form(
        Div(
            Label("Map Template", cls="label"),
            Select(
                *[Option(name, value=name) for name in sorted(templates.keys())],
                name="template_name", cls="select select-bordered w-full"
            ),
            cls="form-control"
        ),
        Div(
            Label("Kingdoms", cls="label"),
            Input(type="number", name="kingdoms", value="5", min="1", max="10",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Lakes", cls="label"),
            Input(type="number", name="lakes", value="1", min="0", max="5",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Div(
            Label("Hex Radius", cls="label"),
            Input(type="number", name="radius", value="25", min="10", max="40",
                  cls="input input-bordered w-full"),
            cls="form-control"
        ),
        Button("Create World", type="submit", cls="btn btn-primary mt-4"),
        action="/create_world", method="post",
        cls="card bg-base-200 shadow-lg p-6 space-y-4 max-w-md mx-auto"
    )
    return Titled("New Game",
        Div(
            H3("Choose Your World", cls="text-2xl font-bold text-center mb-6"),
            form,
            cls="flex flex-col items-center p-12"
        )
    )




In [ ]:
#| export
@patch
def create_game(self: GameStorage, user_id, template_name="bayArea",
                kingdoms=5, lakes=1, radius=10) -> ActiveGame:
    logging.info(f"create_game: user={user_id} template={template_name}")
    tt = TerrainTemplate()
    terrain = getattr(tt, template_name)()
    
    terrain.carve_to_ocean(num_lakes=lakes)
    terrain.hexGrid.adjustRadius(radius)

    # GameBoard now creates cover + basins internally
    board = GameBoard(terrain, top_n=kingdoms)
    board.expand_kingdoms(max_rounds=50)
    
    # Save using the cover that GameBoard created
    board.cover.db = self
    board.cover.save(name=template_name)
    board.save(self, world_id=board.cover.ident)

    self.db.execute("UPDATE user SET activeWorld = ? WHERE id = ?",
                    [board.cover.ident, user_id])
    
    return ActiveGame(board=board, cover=board.cover, world_id=board.cover.ident)


In [ ]:
read_url(url="https://www.fastht.ml/docs/llms-ctx.txt")

## Common Routes

@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    logging.info(f"create world redirecting")
    return RedirectResponse('/game', status_code=303)


In [ ]:
#| export
@rt
def create_world(session, template_name: str, kingdoms: int = 5,
                 lakes: int = 1, radius: int = 10):
    uid = ensure_user(session)
    logging.info(f"create_world: session keys={list(session.keys())}")
    invalidate_cache(uid)  # <-- clear old game
    globalStore.create_game(uid, template_name=template_name,
                            kingdoms=kingdoms, lakes=lakes, radius=radius)
    return RedirectResponse('/game', status_code=303)

In [ ]:
#| export
_game_cache = {}
_cache_lock = threading.Lock()

def invalidate_cache(user_id):
    """Call this when creating a new game or the board changes."""
    with _cache_lock:
        _game_cache.pop(user_id, None)


dummySession= {'userid': 667256 }
webMe(index(dummySession))

In [ ]:
!tail -10 base.text

In [ ]:
#| export
# Get all users
def showUsers():
    users_df = pd.DataFrame(globalStore.users())
    print("Users:")
    print(users_df)

In [ ]:
showUsers()

In [ ]:
dummySession= {'userid': 64801 }
#webMe(index(dummySession))

SVGBuilder.BUILDERHIDE = True

myRessult = globalStore.active_board(64801)
myBoard = myRessult.board
myTerrain = myBoard.terrain
show(myTerrain)

In [ ]:
worldId = 3
k = next((k for k in myBoard.kingdoms if k.countryId == worldId), None)
if k:
    print(k.countryName)
#result = globalStore.kingdom_detail(myRessult.world_id, worldId, myRessult.cover)
#show(result.terrain)

In [ ]:
k.settlements